# cloudposterior: Caching

cloudposterior automatically caches sampling results so you never re-run the same model twice. This is useful even without cloud execution -- just wrap your model in `cp.cloud()` and re-running a notebook cell returns the cached result instantly.

Two caching modes:

- **In-memory** (`cache=True`, the default) -- results are cached for the current session. Re-running a cell in the same kernel is instant.
- **Disk** (`cache="disk"`) -- results persist across kernel restarts. Re-opening a notebook and running the same model returns the cached result without any sampling.

In [1]:
import numpy as np
import pandas as pd
import pymc as pm
import arviz as az

import cloudposterior as cp

## Setup

Using the Radon model from [basics.ipynb](basics.ipynb).

In [2]:
df = pd.read_csv(pm.get_data("radon.csv"))

with pm.Model(name="radon_intercepts", coords={"county": df.county.unique()}) as radon:
    mu_a = pm.Normal("mu_a", mu=0, sigma=5)
    sigma_a = pm.HalfNormal("sigma_a", sigma=2)
    a_raw = pm.Normal("a_raw", mu=0, sigma=1, dims="county")
    a = pm.Deterministic("a", mu_a + sigma_a * a_raw, dims="county")
    b_floor = pm.Normal("b_floor", mu=0, sigma=5)
    mu = a[df.county_code.values] + b_floor * df.floor.values
    sigma_y = pm.HalfNormal("sigma_y", sigma=2)
    pm.Normal("obs", mu=mu, sigma=sigma_y, observed=df.log_radon.values)

## Local caching (no cloud needed)

You don't need cloud execution to use caching. Just wrap your model in `cp.cloud()` -- it intercepts `pm.sample()` and caches the result. Sampling runs locally with PyMC's normal output. The second time you run the same cell, the result is returned from cache.

This is useful when you're iterating on analysis code downstream of sampling -- you don't want to re-sample every time you tweak a plot.

In [3]:
# First run: samples normally (PyMC progress bar shown)
with cp.cloud(radon):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [radon_intercepts::mu_a, radon_intercepts::sigma_a, radon_intercepts::a_raw, radon_intercepts::b_floor, radon_intercepts::sigma_y]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


In [4]:
# Re-run: instant (in-memory cache hit)
with cp.cloud(radon):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

## Disk caching

With `cache="disk"`, results are saved to `.cloudposterior/` and survive kernel restarts. Restart your kernel and re-run this cell -- the result comes back instantly without any sampling.

The cache key includes the model structure, observed data, and all sampling parameters. Changing any of these triggers a new sample.

In [5]:
with cp.cloud(radon, cache="disk"):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [radon_intercepts::mu_a, radon_intercepts::sigma_a, radon_intercepts::a_raw, radon_intercepts::b_floor, radon_intercepts::sigma_y]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 2 seconds.


In [9]:
with cp.cloud(radon, cache="disk"):
    idata = pm.sample(draws=2000, tune=1000, chains=4)

## Cache layout

The disk cache uses human-readable directory names with a hash suffix for uniqueness:

```
.cloudposterior/
├── radon_intercepts/
│   └── draws2000_tune1000_chains4-a3f7b2c9.nc
└── radon_slopes/
    └── draws2000_tune1000_chains4-7c2e5fa8.nc
```

Model names come from `pm.Model(name="radon_intercepts")`. The hash suffix ensures that runs with different non-displayed parameters (like `random_seed`) get separate cache files.

## Model iteration

Caching works naturally with model iteration. Each model variant gets its own cache entry. Switching back to a previous model returns the cached result.

In [6]:
with pm.Model(name="radon_slopes", coords={"county": df.county.unique()}) as radon_slopes:
    mu_a = pm.Normal("mu_a", mu=0, sigma=5)
    sigma_a = pm.HalfNormal("sigma_a", sigma=2)
    a_raw = pm.Normal("a_raw", mu=0, sigma=1, dims="county")
    a = pm.Deterministic("a", mu_a + sigma_a * a_raw, dims="county")
    mu_b = pm.Normal("mu_b", mu=0, sigma=5)
    sigma_b = pm.HalfNormal("sigma_b", sigma=2)
    b_raw = pm.Normal("b_raw", mu=0, sigma=1, dims="county")
    b = pm.Deterministic("b", mu_b + sigma_b * b_raw, dims="county")
    mu = a[df.county_code.values] + b[df.county_code.values] * df.floor.values
    sigma_y = pm.HalfNormal("sigma_y", sigma=2)
    pm.Normal("obs", mu=mu, sigma=sigma_y, observed=df.log_radon.values)

In [7]:
# New model -> samples fresh
with cp.cloud(radon_slopes, cache="disk"):
    idata_slopes = pm.sample(draws=2000, tune=1000, chains=4)

Initializing NUTS using jitter+adapt_diag...
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [radon_slopes::mu_a, radon_slopes::sigma_a, radon_slopes::a_raw, radon_slopes::mu_b, radon_slopes::sigma_b, radon_slopes::b_raw, radon_slopes::sigma_y]


Output()

Sampling 4 chains for 1_000 tune and 2_000 draw iterations (4_000 + 8_000 draws total) took 3 seconds.
There was 1 divergence after tuning. Increase `target_accept` or reparameterize.


In [8]:
az.summary(idata_slopes, filter_vars="like", var_names=["mu_a", "sigma_a", "mu_b", "sigma_b", "sigma_y"])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
radon_slopes::mu_a,1.492,0.052,1.394,1.585,0.001,0.001,2966.0,4775.0,1.0
radon_slopes::mu_b,-0.648,0.083,-0.804,-0.494,0.001,0.001,6024.0,5490.0,1.0
radon_slopes::sigma_a,0.324,0.045,0.238,0.405,0.001,0.001,2828.0,4584.0,1.0
radon_slopes::sigma_b,0.258,0.128,0.007,0.462,0.004,0.001,1270.0,2124.0,1.0
radon_slopes::sigma_y,0.721,0.018,0.688,0.756,0.000,0.000,6691.0,5319.0,1.0
